# 1 — Validation and the gate

*The SemOps Manual — Chapters 8 and 9*

Why an unscoped quality gate dies in its first month, and the one flag that fixes it.
Everything here runs on a fixture the notebook builds itself, so the numbers are
small enough to check by eye — the manual reports the same shape at full scale.

---

## Prerequisites

This notebook is **self-contained**: it writes its own fixtures into a temporary
directory, so nothing needs to exist on disk beforehand. What it does need:

```bash
pip install ontology-quality-suite shacl
```

The next cell checks what is available and prints exactly what is missing.
Cells that need a tool you do not have will say so and skip, rather than
failing with a stack trace.

In [ ]:
import json, os, shutil, subprocess, sys, tempfile, textwrap
from pathlib import Path

WORK = Path(tempfile.mkdtemp(prefix="semops-nb-"))
print("working directory:", WORK)


def have(mod):
    try:
        __import__(mod)
        return True
    except ImportError:
        return False


HAVE_SUITE = have("ontology_suite")
HAVE_SHACL = have("shacl")
SHACL_CLI = shutil.which("shacl")

print("ontology-quality-suite:", "yes" if HAVE_SUITE else "NO  (pip install ontology-quality-suite)")
print("shacl (python)       :", "yes" if HAVE_SHACL else "NO  (pip install shacl)")
print("shacl (cli)          :", SHACL_CLI or "not on PATH")

if HAVE_SHACL:
    import shacl as shacl_py
    print("shacl version        :", getattr(shacl_py, "__version__", "unknown"))


def write(name, text):
    """Write a fixture into the working directory and return its path."""
    p = WORK / name
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(textwrap.dedent(text).lstrip(), encoding="utf-8")
    return p


def suite(*args):
    """Run the ontology suite CLI and show its output."""
    if not HAVE_SUITE:
        print("skipped: ontology-quality-suite is not installed")
        return None
    r = subprocess.run([sys.executable, "-m", "ontology_suite", *map(str, args)],
                       capture_output=True, text=True)
    print(r.stdout.strip() or r.stderr.strip()[:2000])
    return r

## An ontology that imports someone else's

The lesson needs two things: an ontology **you** own, with a couple of real flaws,
and an imported vocabulary you do **not** own, which has its own imperfections.
That is the situation every real project is in.

In [ ]:
upstream = write("vendor/upstream.ttl", """
    @prefix owl:  <http://www.w3.org/2002/07/owl#> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    @prefix up:   <http://upstream.example/ns/> .

    <http://upstream.example/ns/> a owl:Ontology .

    # A perfectly ordinary upstream vocabulary -- note it does not label
    # everything, which is its own business, not ours.
    up:Party      a owl:Class .
    up:Unit       a owl:Class .
    up:memberOf   a owl:ObjectProperty .
""")

ours = write("acme.ttl", """
    @prefix owl:  <http://www.w3.org/2002/07/owl#> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    @prefix acme: <https://acme.example.org/ns/> .
    @prefix up:   <http://upstream.example/ns/> .

    <https://acme.example.org/ns/> a owl:Ontology ;
        owl:imports <http://upstream.example/ns/> .

    acme:Employee   a owl:Class ; rdfs:subClassOf up:Party ;
        rdfs:label "Employee" .

    # FLAW 1: disjoint with its own superclass -> unsatisfiable (LOG-001)
    acme:Contractor a owl:Class ; rdfs:subClassOf acme:Employee ;
        rdfs:label "Contractor" ;
        owl:disjointWith acme:Employee .

    # FLAW 2: no label at all (QUA-001)
    acme:hasSkill   a owl:DatatypeProperty .

    # FLAW 3: not lowerCamelCase, and no domain or range (STY-002, STR-003)
    acme:reports_to a owl:ObjectProperty ; rdfs:label "reports to" .
""")
print(ours.read_text(encoding="utf-8")[:400], "...")

## The unscoped run

The obvious command. Resolve the imports, run the whole registry, see what comes back.

In [ ]:
r = suite("checks", "--ontology", ours, "--import-dir", WORK / "vendor",
          "--engine", "sparql",
          "--out-dir", WORK / "out/all", "--fail-on", "never")

Findings against terms **we** did not write are in there too — the upstream
vocabulary is part of the merged graph, so the registry checks it as thoroughly
as it checks ours. On a real vocabulary such as FOAF or the W3C Organization
Ontology that is where the count runs to several hundred.

## Scoping to what we own

Keep the imports resolved — we still want `up:Party` to genuinely exist — and
filter the **report** to findings whose focus node is in our namespace.

In [ ]:
r = suite("checks", "--ontology", ours, "--import-dir", WORK / "vendor",
          "--engine", "sparql",
          "--own-namespace", "https://acme.example.org/ns/",
          "--out-dir", WORK / "out/own", "--fail-on", "never")

### The near-miss that returns nothing

`--own-namespace` is a **literal IRI-prefix string match**. Get the string
slightly wrong and you get a confidently empty report — which looks exactly
like a passing gate. This is worth doing once, deliberately, so you recognise it.

In [ ]:
# Note the http (not https) and the # instead of the trailing slash.
r = suite("checks", "--ontology", ours, "--import-dir", WORK / "vendor",
          "--engine", "sparql",
          "--own-namespace", "http://acme.example.org/ns#",
          "--out-dir", WORK / "out/typo", "--fail-on", "never")
print("\n^ zero findings, and nothing said the filter matched nothing.")

### The wrong fix

The instinctive alternative is to stop resolving imports altogether. It is
quieter, and it is worse: terms defined upstream now look undefined, so you
trade irrelevant findings for **false** ones.

In [ ]:
r = suite("checks", "--ontology", ours, "--exclude-imports",
          "--engine", "sparql",
          "--out-dir", WORK / "out/excl", "--fail-on", "never")
print("\n^ look for DAT-002 / dangling-reference findings pointing at up: terms.")
print("  They are artefacts of not loading the import, not defects in our file.")

## The gate

`--fail-on Violation` is the whole mechanism: a non-zero exit when a finding at
or above that severity exists.

In [ ]:
r = suite("checks", "--ontology", ours, "--import-dir", WORK / "vendor",
          "--engine", "sparql",
          "--own-namespace", "https://acme.example.org/ns/",
          "--out-dir", WORK / "out/gate", "--fail-on", "Violation")
if r: print("\nexit code:", r.returncode, "-> a CI job would", "FAIL" if r.returncode else "pass")

---

## What to take away

| | |
|---|---|
| Unscoped, imports resolved | Correct, and unusable as a gate — most findings are not yours |
| `--exclude-imports` | Quieter, but introduces false findings |
| `--own-namespace <your IRI prefix>` | Scoped to what you own, imports still real |

Copy the namespace from your ontology's `@prefix` line rather than typing it,
and keep one deliberately-failing fixture in CI so a gate that *cannot* fail is
detectable.

Next: [2 — Rules and inference](02-rules-and-inference.ipynb).